# NBA 3.1 Sample From Combined Physician Data

Cuts the full combined physician table (`NBA_3_1_Combine_Physician_Data`, all HCPs x c1-c28) down to the rows of a sample list (`Target_Variable/HCP_Target_Variable`, default `all_hcps_5_train_months`) and adds the six sample columns. The result and a copy of the feature dictionary are written to `hcp_combined/_sample_<SAMPLE_NAME>/`, the folder the feature selection notebooks read.

Feature values are copied from the full table unchanged, so they are the same as in the full run (imputation statistics from all HCPs).
**Widgets:** `RUN_NAME` (full combined run folder), `SAMPLE_NAME` (sample list in `sample_lists/`).

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("RUN_NAME", "combined_2026_11_26_full")
dbutils.widgets.text("SAMPLE_NAME", "all_hcps_5_train_months")
RUN_NAME = dbutils.widgets.get("RUN_NAME").strip()
SAMPLE_NAME = dbutils.widgets.get("SAMPLE_NAME").strip()
assert RUN_NAME and SAMPLE_NAME

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
DEV = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
FULL_DIR = f"{DEV}{RUN_NAME}/hcp_combined/"
FULL_PATH = FULL_DIR + "kerendia_nba_3_1_hcp_selected_features_data"
FULL_DICTIONARY = FULL_DIR + "kerendia_nba_3_1_hcp_feature_dictionary"
SAMPLE_LIST_PATH = f"{VOL31}hcp_level_model/dev_pipeline/sample_lists/{SAMPLE_NAME}"
OUT_DIR = f"{FULL_DIR}_sample_{SAMPLE_NAME}/"
OUT_PATH = OUT_DIR + "kerendia_nba_3_1_hcp_selected_features_data"
OUT_DICTIONARY = OUT_DIR + "kerendia_nba_3_1_hcp_feature_dictionary"
assert OUT_DIR.startswith(VOL31) and "kerendia_nba_3_volume" not in OUT_DIR

ID_COLS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE"]
SAMPLE_COLS = ["SPLIT", "SAMPLE_GROUP", "IS_EXISTING_WRITER", "WEIGHT_GROUP", "WEIGHT_MONTH", "WEIGHT"]
print("Full table :", FULL_PATH)
print("Sample list:", SAMPLE_LIST_PATH)
print("Output     :", OUT_DIR)

In [0]:
full = spark.read.format("delta").load(FULL_PATH)
sample = spark.read.format("delta").load(SAMPLE_LIST_PATH).select("HCP_COHORT_ID", *SAMPLE_COLS)
n_sample = sample.count()
assert sample.select("HCP_COHORT_ID").distinct().count() == n_sample, "duplicate rows in the sample list"
FEATURE_COLS = [c for c in full.columns if c not in ID_COLS]
assert not set(FEATURE_COLS) & set(SAMPLE_COLS), "a feature has the same name as a sample column"

# Same column layout as a Combine run with SAMPLE_LIST: ID columns, sample columns, features
out = (full.join(F.broadcast(sample) if n_sample < 1_000_000 else sample, "HCP_COHORT_ID", "inner")
           .select(*ID_COLS, *SAMPLE_COLS, *[F.col(f"`{c}`") for c in FEATURE_COLS]))
out.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUT_PATH)
(spark.read.format("delta").load(FULL_DICTIONARY)
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUT_DICTIONARY))
print(f"Saved {spark.read.format('delta').load(OUT_PATH).count():,} rows -> {OUT_PATH}")

In [0]:
# Every sample row present once, sample columns as in the list, features identical to the full table,
# dictionary = feature columns, no empty values in a few spread features
s = spark.read.format("delta").load(OUT_PATH)
n = s.count()
assert n == n_sample, (n, n_sample)
assert s.select("HCP_COHORT_ID").distinct().count() == n, "duplicate HCP_COHORT_ID"
assert s.columns == ID_COLS + SAMPLE_COLS + FEATURE_COLS, "column layout differs"

sl = spark.read.format("delta").load(SAMPLE_LIST_PATH)
diff = s.select("HCP_COHORT_ID", *SAMPLE_COLS).exceptAll(sl.select("HCP_COHORT_ID", *SAMPLE_COLS)).count()
assert diff == 0, f"{diff} rows with sample columns different from the list"

dictionary = spark.read.format("delta").load(OUT_DICTIONARY).toPandas()
assert set(dictionary.FEATURE_NAME) == set(FEATURE_COLS), "dictionary and feature columns differ"

check = FEATURE_COLS[::max(1, len(FEATURE_COLS) // 20)]   # 20 spread features
keys = s.select("HCP_COHORT_ID").sample(fraction=0.01, seed=42)
mism = (s.join(keys, "HCP_COHORT_ID").select("HCP_COHORT_ID", *[F.col(f"`{c}`") for c in check])
         .exceptAll(full.join(keys, "HCP_COHORT_ID").select("HCP_COHORT_ID", *[F.col(f"`{c}`") for c in check])).count())
assert mism == 0, f"{mism} rows differ from the full table"
nulls = s.select(*[F.sum(F.col(f"`{c}`").isNull().cast("int")) for c in check]).first()
assert all(v == 0 for v in nulls), "empty values"

print(f"{n:,} rows, {s.select('PROVIDER_ID').distinct().count():,} HCPs, {len(FEATURE_COLS):,} features - checks passed")
(s.groupBy("SPLIT", "COHORT_SET").agg(F.count("*").alias("rows"), F.sum(F.col("IS_EXISTING_WRITER").cast("int")).alias("writer_rows"))
  .orderBy("SPLIT", "COHORT_SET").display())